In [1]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "f_ILeunxSxcQbovCWUSjwFCoyERPGBQVsnny"
os.environ["MEGALLM_API_KEY"] = "sk-mega-691b408a0f44f380720dd85cf734fc9080694884a6a57a0e4321d03fa93e13c9"

In [2]:
!pip install -U langchain langchain-core langchain-community chromadb faiss-cpu sentence-transformers wikipedia

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.13
    Uninstalling langchain-core-1.2.13:
      Successfully uninstalled langchain-core-1.2.13
  Attempting uninstall: sentence-transformers━━━ 0/5 [langchain-core]
    Found existing installation: sentence-transformers 5.2.2m [langchain-core]
    Uninstalling sentence-transformers-5.2.2: 0/5 [langchain-core]
      Successfully uninstalled sentence-transformers-5.2.2 [langchain-core]
  Attempting uninstall: langgraph-prebuilt━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [sentence-transformers]
    Found existing installation: langgraph-prebuilt 1.0.7━━━━━ 1/5 [sentence-transformers]
    Uninstalling langgraph-prebuilt-1.0.7:━━━━━━━━━━━━━━━━━━━━ 1/5 [sentence-transformers]
      Successfully uninstalled langgraph-prebuilt-1.0.7━━━━━━━ 1/5 [sentence-transformers]
  Attempting uninstall: langgraph━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [sentence-transformers]
    Found existing installation: langgraph 1.0.8━━━━━━━━━━━━━━

## Wikipedia Retriever

In [3]:
from langchain_community.retrievers import WikipediaRetriever

In [4]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [5]:

# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [6]:
docs

[Document(metadata={'title': 'India–Pakistan wars and conflicts', 'summary': 'Since the partition of British India in 1947 and subsequent creation of the dominions of India and Pakistan, the two countries have been involved in a number of wars, conflicts, and military standoffs. A long-running dispute over Kashmir and cross-border terrorism have been the predominant cause of conflict between the two states, with the exception of the Indo-Pakistani War of 1971, which occurred as a direct result of hostilities stemming from the Bangladesh Liberation War in erstwhile East Pakistan (now Bangladesh).', 'source': 'https://en.wikipedia.org/wiki/India%E2%80%93Pakistan_wars_and_conflicts'}, page_content='Since the partition of British India in 1947 and subsequent creation of the dominions of India and Pakistan, the two countries have been involved in a number of wars, conflicts, and military standoffs. A long-running dispute over Kashmir and cross-border terrorism have been the predominant caus

In [7]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Since the partition of British India in 1947 and subsequent creation of the dominions of India and Pakistan, the two countries have been involved in a number of wars, conflicts, and military standoffs. A long-running dispute over Kashmir and cross-border terrorism have been the predominant cause of conflict between the two states, with the exception of the Indo-Pakistani War of 1971, which occurred as a direct result of hostilities stemming from the Bangladesh Liberation War in erstwhile East Pakistan (now Bangladesh).


== Background ==

The Partition of India came in 1947 with the sudden grant of independence. It was the intention of those who wished for a Muslim state to have a clean partition between independent and equal "Pakistan" and "Hindustan" once independence came.
Nearly one third of the Muslim population of India remained in the new India.
Inter-communal violence between Hindus, Sikhs and Muslims resulted in between 200,000 and 2 million casualti

## Vector Store Retriever

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [9]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [10]:
# Step 2: Initialize embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="my_collection"
)

/var/folders/mg/8srxqngd5nscy1jhw2q5rjb80000gn/T/ipykernel_88066/760115664.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/Users/apple/Documents/GenAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [12]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [13]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [14]:
results = vectorstore.similarity_search(query, k=2)

In [15]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


## MMR

In [16]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [17]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [18]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [19]:
query = "What is langchain?"
results = retriever.invoke(query)

In [20]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 3 ---
MMR helps you get diverse results when doing similarity search.


## Multiquery Retriever

In [26]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

In [27]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [28]:
# Initialize OpenAI embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embeddings)

In [29]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [30]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm = ChatOpenAI(
    base_url="https://ai.megallm.io/v1",
    api_key=os.getenv("MEGALLM_API_KEY"),
    model="openai-gpt-oss-20b"
)
)

In [31]:
# Query
query = "How to improve energy levels and maintain balance?"

In [32]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [33]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 5 ---
Deep sleep is crucial for cellular repair and emotional regulation.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improv

## ContextualCompressionRetriever

In [34]:
from langchain_community.vectorstores import FAISS
from langchain_openai import  ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [35]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [36]:
# Create a FAISS vector store from the documents
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)
vectorstore = FAISS.from_documents(docs, embeddings)

In [37]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [38]:
# Set up the compressor using an LLM
llm = ChatOpenAI(
    base_url="https://ai.megallm.io/v1",
    api_key=os.getenv("MEGALLM_API_KEY"),
    model="openai-gpt-oss-20b"
)
compressor = LLMChainExtractor.from_llm(llm)

In [39]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [40]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [41]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.

--- Result 3 ---
Photosynthesis does not occur in animal cells.
